In [ ]:
path_pattern = "/workspaces/mads-siads-699-winter-2026-capstone/notebooks/data/source_data/initial_dataset/*.parquet"

In [ ]:
import polars as pl
import glob

all_files = sorted(glob.glob(path_pattern))
first_x = all_files[3:10]
if first_x:
    df_deps = pl.read_parquet(first_x, n_rows=100)
    print(f"Loaded {len(first_x)} files.")

In [ ]:
import requests
from datetime import datetime
session = requests.Session()
session.mount("https://", requests.adapters.HTTPAdapter(max_retries=3))

def get_metrics(row):
    dep_pkg = row["dep_name"]
    dep_ver = row["dep_version"]
    snapshot_date = row["SnapshotAt"]

    try:
        # PyPI
        res = session.get(f"https://pypi.org/pypi/{dep_pkg}/json", timeout=10)
        if res.status_code != 200:
            return {"ttu_days": None, "ttr_days": None}
            
        data = res.json()
        releases = data.get('releases', {})

        latest_ver = data['info']['version']
        
        # TTU
        latest_release_date = datetime.fromisoformat(releases[latest_ver][0]['upload_time'].replace("Z", ""))
        ttu = max(0, (latest_release_date - snapshot_date).days)

        #TTR
        ttr = None
        osv_query = {"version": dep_ver, "package": {"name": dep_pkg, "ecosystem": "PyPI"}}
        osv_res = session.post("https://api.osv.dev/v1/query", json=osv_query, timeout=10).json()
        
        if 'vulns' in osv_res:
            vuln = osv_res['vulns'][0]
            pub_date = datetime.fromisoformat(vuln['published'].replace("Z", ""))
            for affected in vuln.get('affected', []):
                for event in affected.get('ranges', [{}])[0].get('events', []):
                    if 'fixed' in event and event['fixed'] in releases:
                        fix_date = datetime.fromisoformat(releases[event['fixed']][0]['upload_time'].replace("Z", ""))
                        ttr = max(0, (fix_date - pub_date).days)
                        break
        
        return {"ttu_days": ttu, "ttr_days": ttr}
    except Exception as e:
        print(f"Error occurred while processing {dep_pkg}: {e}")
        return {"ttu_days": None, "ttr_days": None}

In [ ]:
df_results = df_deps.with_columns(
    pl.struct(["dep_name", "dep_version", "SnapshotAt"])
    .map_elements(lambda x: get_metrics(x), return_dtype=pl.Struct([
        pl.Field("ttu_days", pl.Int64),
        pl.Field("ttr_days", pl.Int64)
    ]))
    .alias("metrics")
).unnest("metrics")

In [ ]:
mttu_df = df_results.group_by("package_name").agg(
    pl.mean("ttu_days").alias("mttu"),
)
mttu_df.head(30)

In [ ]:
output_dir = "./data/source_data/ttu_ttr/"

df_results.write_parquet(
    output_dir,
    partition_by="package_name"
)

In [ ]:
path_pattern = "/workspaces/mads-siads-699-winter-2026-capstone/notebooks/data/source_data/pypi_scorecards/*.parquet"
all_files = sorted(glob.glob(path_pattern))
first_x = all_files[:10]
if first_x:
    df_scorecard = pl.read_parquet(first_x)
    print(f"Loaded {len(first_x)} files.")

In [ ]:
df_deps = df_deps.with_columns(
    pl.col("github_repo").str.replace("https://", "").str.replace("github.com/", ""),
    pl.col("package_published_at").cast(pl.Datetime)
)

dtype = df_scorecard.schema.get("scorecard_date")
if dtype is not None and "Date" in str(dtype):
    df_scorecard = df_scorecard.with_columns(
        pl.col("repo_name").str.replace("https://", "").str.replace("github.com/", ""),
        pl.col("scorecard_date").cast(pl.Datetime)
    )
else:
    df_scorecard = df_scorecard.with_columns(
        pl.col("repo_name").str.replace("https://", "").str.replace("github.com/", ""),
        pl.col("scorecard_date").str.to_datetime().cast(pl.Datetime)
    )


df_deps = df_deps.sort("package_published_at")
df_scorecard = df_scorecard.sort("scorecard_date")


df_joined = df_deps.join_asof(
    df_scorecard,
    left_on="package_published_at",
    right_on="scorecard_date",
    by_left="github_repo",
    by_right="repo_name",
    strategy="backward"
)

In [ ]:
# df_scorecard = df_scorecard.with_columns(
#     pl.col("repo_name").str.replace(r"^github\.com/", "").alias("match_repo")
# )
# df_deps = df_deps.with_columns(
#     pl.col("github_repo").alias("match_repo")
# )

# df_scorecard_collapsed = (
#     df_scorecard
#     .group_by("match_repo")
#     .agg([
#         pl.col("aggregate_score").first(),  # Keep the main score
#         # Optional: Get specific scores as columns
#         pl.col("check_score").filter(pl.col("check_name") == "Code-Review").first().alias("score_code_review"),
#         pl.col("check_score").filter(pl.col("check_name") == "Maintained").first().alias("score_maintained")
#     ])
# )

# joined_df = df_deps.join(df_scorecard_collapsed, on="match_repo", how="left")